# 03. 과적합 진단과 대응 — 데이터 증강·Dropout·콜백

**과적합(overfitting)** 은 모델이 학습 데이터에만 잘 맞고 새 데이터에는 약해지는 현상입니다.
학습 정확도는 계속 오르는데 검증 정확도가 정체·하락하면 과적합 신호입니다.

이 노트북에서는 과적합을 **눈으로 확인**하고, 대표적 대응책의 효과를 비교합니다.

1. 과적합이 일어나는 모델을 학습해 학습/검증 곡선으로 진단
2. **데이터 증강 + Dropout + EarlyStopping 콜백** 을 적용해 완화
3. 두 경우의 곡선을 비교

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
print('TensorFlow', tf.__version__)

## 1. 데이터 준비

Fashion-MNIST를 사용합니다. 과적합을 빠르게 관찰하기 위해 학습 데이터를 일부만 사용합니다(데이터가 적을수록 과적합이 잘 나타남).

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train = (x_train / 255.0).astype('float32')[..., None]   # (N,28,28) -> (N,28,28,1)
x_test  = (x_test  / 255.0).astype('float32')[..., None]

# 과적합을 잘 보이게 학습 데이터를 5000장으로 축소
x_small, y_small = x_train[:5000], y_train[:5000]
print('축소 학습셋:', x_small.shape)

## 2. 과적합 모델 (대응책 없음)

증강도 Dropout도 없는 모델을 충분히 오래 학습하면, 학습 정확도는 높지만 검증 정확도와 격차가 벌어집니다.

In [ ]:
def base_cnn():
    return keras.Sequential([
        keras.layers.Input((28, 28, 1)),
        keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        keras.layers.MaxPooling2D(),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(10, activation='softmax'),
    ])

model_of = base_cnn()
model_of.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_of = model_of.fit(x_small, y_small, validation_data=(x_test, y_test),
                       epochs=20, batch_size=128, verbose=2)

## 3. 대응: 데이터 증강 + Dropout + EarlyStopping

- **데이터 증강 레이어**(RandomFlip/Rotation/Zoom): 매번 다른 변형을 보여줘 일반화 향상
- **Dropout**: 학습 중 일부 뉴런을 꺼 과의존을 막음
- **EarlyStopping 콜백**: 검증 손실이 더 나아지지 않으면 학습을 자동 중단하고 최적 가중치 복원

In [ ]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
])

model_reg = keras.Sequential([
    keras.layers.Input((28, 28, 1)),
    data_augmentation,
    keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax'),
])

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True)

model_reg.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_reg = model_reg.fit(x_small, y_small, validation_data=(x_test, y_test),
                         epochs=40, batch_size=128, callbacks=[early_stop], verbose=2)

## 4. 비교

두 모델의 학습/검증 정확도 곡선을 나란히 봅니다.
대응책을 적용한 쪽이 학습-검증 격차가 줄고 검증 정확도가 더 안정적인 것을 확인할 수 있습니다.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_of.history['accuracy'], label='학습')
ax1.plot(hist_of.history['val_accuracy'], label='검증')
ax1.set_title('대응 없음 (과적합)'); ax1.set_xlabel('epoch'); ax1.set_ylim(0, 1); ax1.legend()
ax2.plot(hist_reg.history['accuracy'], label='학습')
ax2.plot(hist_reg.history['val_accuracy'], label='검증')
ax2.set_title('증강+Dropout+EarlyStopping'); ax2.set_xlabel('epoch'); ax2.set_ylim(0, 1); ax2.legend()
plt.tight_layout(); plt.show()

### 정리

- 데이터가 적은 모델을 오래 학습하면 **학습-검증 정확도 격차**로 과적합을 진단할 수 있습니다.
- 데이터 증강·Dropout·EarlyStopping은 과적합을 완화하는 대표적 도구이며, Keras에서 레이어·콜백으로 간단히 적용됩니다.
- 실무에서는 이런 정규화 기법을 조합해 일반화 성능을 끌어올립니다.